# LangChain 核心模块学习：Chains

对于简单的大模型应用，单独使用语言模型（LLMs）是可以的。

**但更复杂的大模型应用需要将 `LLMs` 和 `Chat Models` 链接在一起 - 要么彼此链接，要么与其他组件链接。**

LangChain 为这种“链式”应用程序提供了 `Chain` 接口。

LangChain 以通用方式定义了 `Chain`，它是对组件进行调用序列的集合，其中可以包含其他链。

In [1]:
# 去掉pip install 使用项目依赖版本
# ! pip install -U langchain

## Chain Class 基类

类继承关系：

```
Chain --> <name>Chain  # Examples: LLMChain, MapReduceChain, RouterChain
```

**代码实现：https://github.com/langchain-ai/langchain/blob/master/libs/langchain/langchain/chains/base.py**

```python
# 定义一个名为Chain的基础类
class Chain(Serializable, Runnable[Dict[str, Any], Dict[str, Any]], ABC):
    """为创建结构化的组件调用序列的抽象基类。
    
    链应该用来编码对组件的一系列调用，如模型、文档检索器、其他链等，并为此序列提供一个简单的接口。
    
    Chain接口使创建应用程序变得容易，这些应用程序是：
    - 有状态的：给任何Chain添加Memory可以使它具有状态，
    - 可观察的：向Chain传递Callbacks来执行额外的功能，如记录，这在主要的组件调用序列之外，
    - 可组合的：Chain API足够灵活，可以轻松地将Chains与其他组件结合起来，包括其他Chains。
    
    链公开的主要方法是：
    - `__call__`：链是可以调用的。`__call__`方法是执行Chain的主要方式。它将输入作为一个字典接收，并返回一个字典输出。
    - `run`：一个方便的方法，它以args/kwargs的形式接收输入，并将输出作为字符串或对象返回。这种方法只能用于一部分链，不能像`__call__`那样返回丰富的输出。
    """

    # 调用链
    def invoke(
        self, input: Dict[str, Any], config: Optional[runnableConfig] = None
    ) -> Dict[str, Any]:
        """传统调用方法。"""
        return self(input, **(config or {}))

    # 链的记忆，保存状态和变量
    memory: Optional[BaseMemory] = None
    """可选的内存对象，默认为None。
    内存是一个在每个链的开始和结束时被调用的类。在开始时，内存加载变量并在链中传递它们。在结束时，它保存任何返回的变量。
    有许多不同类型的内存，请查看内存文档以获取完整的目录。"""

    # 回调，可能用于链的某些操作或事件。
    callbacks: Callbacks = Field(default=None, exclude=True)
    """可选的回调处理程序列表（或回调管理器）。默认为None。
    在对链的调用的生命周期中，从on_chain_start开始，到on_chain_end或on_chain_error结束，都会调用回调处理程序。
    每个自定义链可以选择调用额外的回调方法，详细信息请参见Callback文档。"""

    # 是否详细输出模式
    verbose: bool = Field(default_factory=_get_verbosity)
    """是否以详细模式运行。在详细模式下，一些中间日志将打印到控制台。默认值为`langchain.verbose`。"""

    # 与链关联的标签
    tags: Optional[List[str]] = None
    """与链关联的可选标签列表，默认为None。
    这些标签将与对这个链的每次调用关联起来，并作为参数传递给在`callbacks`中定义的处理程序。
    你可以使用这些来例如识别链的特定实例与其用例。"""

    # 与链关联的元数据
    metadata: Optional[Dict[str, Any]] = None
    """与链关联的可选元数据，默认为None。
    这些元数据将与对这个链的每次调用关联起来，并作为参数传递给在`callbacks`中定义的处理程序。
    你可以使用这些来例如识别链的特定实例与其用例。"""
```

In [2]:
# 

## LLMChain

LLMChain 是 LangChain 中最简单的链，作为其他复杂 Chains 和 Agents 的内部调用，被广泛应用。

一个LLMChain由PromptTemplate和语言模型（LLM or Chat Model）组成。它使用直接传入（或 memory 提供）的 key-value 来规范化生成 Prompt Template（提示模板），并将生成的 prompt （格式化后的字符串）传递给大模型，并返回大模型输出。

![](../images/llm_chain.png)

In [ ]:
# from langchain_openai import OpenAI
# from langchain_core.prompts import PromptTemplate

# llm = OpenAI(model_name="gpt-3.5-turbo-instruct", temperature=0.9, max_tokens=500)

In [ ]:
# 【新增】ChatOpenAI调用方式，使用gpt-4o-mini模型
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.9, max_tokens=500)

In [5]:
prompt = PromptTemplate(
    input_variables=["product"],
    template="给制造{product}的有限公司取10个好名字，并给出完整的公司名称",
)

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()
print(chain.invoke({
    'product': "性能卓越的GPU"
    }))

In [ ]:
# Verbose mode is handled differently in LCEL, you can use RunnableConfig
# chain.verbose = True  # This is no longer needed with LCEL

In [ ]:
# Verbose is no longer a property of LCEL chains
# chain.verbose

In [ ]:
print(chain.invoke({
    'product': "性能卓越的GPU"
    }))

## Sequential Chain

串联式调用语言模型（将一个调用的输出作为另一个调用的输入）。

顺序链（Sequential Chain ）允许用户连接多个链并将它们组合成执行特定场景的流水线（Pipeline）。有两种类型的顺序链：

- SimpleSequentialChain：最简单形式的顺序链，每个步骤都具有单一输入/输出，并且一个步骤的输出是下一个步骤的输入。
- SequentialChain：更通用形式的顺序链，允许多个输入/输出。

### 使用 SimpleSequentialChain 实现戏剧摘要和评论（单输入/单输出）

![](../images/simple_sequential_chain_0.png)

In [ ]:
# 这是一个链，用于根据剧目的标题撰写简介。
# 【新增】ChatOpenAI调用方式，使用gpt-4o-mini模型
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.9, max_tokens=500)

template = """你是一位剧作家。根据戏剧的标题，你的任务是为该标题写一个简介。

标题：{title}
剧作家：以下是对上述戏剧的简介："""

prompt_template = PromptTemplate(input_variables=["title"], template=template)
synopsis_chain = prompt_template | llm | StrOutputParser()

In [ ]:
# 这是一个链，用于根据剧情简介撰写一篇戏剧评论。
from langchain_core.prompts import PromptTemplate

template = """你是《纽约时报》的戏剧评论家。根据剧情简介，你的工作是为该剧撰写一篇评论。

剧情简介：
{synopsis}

以下是来自《纽约时报》戏剧评论家对上述剧目的评论："""

prompt_template = PromptTemplate(input_variables=["synopsis"], template=template)
review_chain = prompt_template | llm | StrOutputParser()

![](../images/simple_sequential_chain_1.png)

In [ ]:
# 使用 LCEL 的管道运算符将两个链串联起来
# 第一个链的输出会自动成为第二个链的输入，以 'synopsis' 作为键
from langchain_core.runnables import RunnablePassthrough

def create_synopsis_input(title):
    """将标题包装为字典格式"""
    return {"title": title}

def wrap_synopsis_for_review(synopsis):
    """将简介包装为评论链所需的格式"""
    return {"synopsis": synopsis}

overall_chain = (
    RunnablePassthrough() 
    | synopsis_chain 
    | wrap_synopsis_for_review 
    | review_chain
)

In [ ]:
review = overall_chain.invoke({"title": "三体人不是无法战胜的"})
print(review)

In [ ]:
review = overall_chain.invoke({"title": "星球大战第九季"})
print(review)

### 使用 SequentialChain 实现戏剧摘要和评论（多输入/多输出）

![](../images/sequential_chain_0.png)

In [ ]:
# 这是一个链，根据剧名和设定的时代来撰写剧情简介。
# 【新增】ChatOpenAI调用方式，使用gpt-4o-mini模型
from langchain_core.prompts import PromptTemplate

llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.9, max_tokens=500)

template = """你是一位剧作家。根据戏剧的标题和设定的时代，你的任务是为该标题写一个简介。

标题：{title}
时代：{era}
剧作家：以下是对上述戏剧的简介："""

prompt_template = PromptTemplate(input_variables=["title", "era"], template=template)
synopsis_chain = prompt_template | llm | StrOutputParser()

In [ ]:
# 这是一个链，用于根据剧情简介撰写一篇戏剧评论。
from langchain_core.prompts import PromptTemplate

template = """你是《纽约时报》的戏剧评论家。根据该剧的剧情简介，你需要撰写一篇关于该剧的评论。

剧情简介：
{synopsis}

来自《纽约时报》戏剧评论家对上述剧目的评价："""

prompt_template = PromptTemplate(input_variables=["synopsis"], template=template)
review_chain = prompt_template | llm | StrOutputParser()

In [ ]:
# 使用 LCEL 构建多输入/多输出的链
# 使用 RunnablePassthrough 和字典来保留中间结果
from langchain_core.runnables import RunnablePassthrough

def add_synopsis_to_dict(x):
    """将简介添加到输入字典中"""
    return {**x, "synopsis": x["synopsis"]}

m_overall_chain = (
    RunnablePassthrough.assign(synopsis=synopsis_chain)
    | RunnablePassthrough.assign(review=lambda x: review_chain.invoke({"synopsis": x["synopsis"]}))
)

In [ ]:
result = m_overall_chain.invoke({"title":"三体人不是无法战胜的", "era": "二十一世纪的新中国"})
print(f"标题: {result['title']}")
print(f"时代: {result['era']}")
print(f"\n简介:\n{result['synopsis']}")
print(f"\n评论:\n{result['review']}")

In [ ]:
#### The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
### 【新增】langchain 0.3版本，使用RunnableSequence替换LLMChain，并指定 output_key

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableSequence

llm = ChatOpenAI(
    model_name="gpt-3.5-turbo",
    temperature=0,
    max_tokens=200
)

summarizing_prompt_template = """
总结以下文本为一个 20 字以内的句子:
---
{content}
"""
summarizing_prompt = PromptTemplate.from_template(summarizing_prompt_template)
summarizing_chain = summarizing_prompt | llm | StrOutputParser()

translating_prompt_template = """
将{summary}翻译成英文:
"""
translating_prompt = PromptTemplate.from_template(translating_prompt_template)
translating_chain = translating_prompt | llm | StrOutputParser()

# Construct a RunnableSequence with custom output keys
overall_chain = summarizing_chain | {
    'summary': summarizing_chain,
    'translation': translating_chain
}

test_contetent = """
端到端在深度学习中指的是一种模型架构设计理念：
从原始输入数据到最终输出结果，整个决策过程完全由单一神经网络完成，无需人工设计中间处理环节。
这种设计摒弃了传统分步骤、模块化的处理流程，让模型自主挖掘数据中隐藏的复杂关联。
"""

result = overall_chain.invoke({"content": test_contetent})
print(result)

### Homework

#### 使用 OutputParser 优化 overall_chain 输出格式，区分 synopsis_chain 和 review_chain 的结果